# GP05 — CRS Without Fear: Ενδεικτική λύση

Το workflow κρατά το αρχικό geographic CRS και δημιουργεί νέο projected GeoDataFrame για μετρήσεις.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
data_path = Path('data/gp05/greek_sites.csv')
df = pd.read_csv(data_path)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['longitude'], df['latitude']), crs='EPSG:4326')


In [ ]:
print('CRS:', gdf.crs)
print('rows:', len(gdf))
print('geometry types:', gdf.geom_type.value_counts().to_dict())
print('bounds:', gdf.total_bounds)


In [ ]:
projected = gdf.to_crs('EPSG:2100')
print('projected CRS:', projected.crs)
print('projected bounds:', projected.total_bounds)


Το `set_crs` δηλώνει την ερμηνεία των υπαρχουσών συντεταγμένων. Το `to_crs` μετασχηματίζει τις γεωμετρίες. Εδώ χρησιμοποιούμε `to_crs`, επειδή οι μοίρες πρέπει να γίνουν μέτρα. Το `wrong` παρακάτω είναι ελεγχόμενο παράδειγμα: τεχνικά έχει CRS, αλλά τα bounds αποκαλύπτουν ότι degrees βαφτίστηκαν ως metres.

In [ ]:
wrong = gdf.set_crs('EPSG:2100', allow_override=True)
assert gdf.crs.to_epsg() == 4326
assert projected.crs.to_epsg() == 2100
assert len(projected) == len(gdf)
assert projected.geom_type.eq('Point').all()
assert projected.geometry.notna().all()
print('wrong declared bounds:', wrong.total_bounds)
print('correct projected bounds:', projected.total_bounds)


In [ ]:
athens = projected.loc[projected['name'].eq('Athens'), 'geometry'].iloc[0]
thessaloniki = projected.loc[projected['name'].eq('Thessaloniki'), 'geometry'].iloc[0]
distance_m = athens.distance(thessaloniki)
print(f'Athens–Thessaloniki: {distance_m:,.0f} m ({distance_m / 1000:.1f} km)')
assert 300_000 < distance_m < 600_000
